# Datos Masivos — McCain: ¿dónde crecerá el mercado de papas a la francesa?

**Pipeline completo + modelo XGBoost + heatmap interactivo, en Databricks**

El notebook 04 responde **dónde invertir hoy** (índice de 0 a 100). Este notebook agrega la
pregunta que sigue: **¿dónde crecerá más el mercado en los próximos años y cuánto vale ese
crecimiento?**

1. Corremos el pipeline completo con una sola llamada (`ejecutar_pipeline`), como en la clase de
   automatización: adquisición → Silver → validación → Gold → modelo. Si la validación falla, no
   se construye Gold.
2. Un modelo **XGBoost** aprende de cómo crecieron los restaurantes de cada municipio entre las
   versiones del DENUE (2015 → 2019 → 2022 → 2026) y proyecta 2026 → 2030.
3. El **potencial en US$** de cada metrópoli es restaurantes proyectados × kg de papa congelada
   importada por restaurante × US$/kg (UN Comtrade). Es tamaño de mercado, **no utilidad**: no
   tenemos ventas, márgenes ni costos.

**Cómo correrlo:** Serverless y **Run all** (unos 5 minutos: lo descargado ya está en caché).
También se puede programar como Job: los widgets de arriba son sus parámetros.

## 0. Instalar librerías

In [ ]:
%pip install -q xgboost beautifulsoup4 pyyaml

In [ ]:
dbutils.library.restartPython()

## 1. Parámetros (widgets)

| Widget | Qué hace |
|---|---|
| `descargar` | `si`: revisa las fuentes (lo que ya está se toma de la caché). `no`: usa solo `raw/`. |
| `registrar_tablas` | `si`: registra las tablas Gold como Delta en `workspace.mccain_papa`. |
| `exportar_muestra` | `si`: guarda una muestra real chica del DENUE y Comtrade para el repositorio. |
| `metropolis_muestra` | Las metrópolis de esa muestra, separadas por coma. |

In [ ]:
dbutils.widgets.dropdown("descargar", "si", ["si", "no"])
dbutils.widgets.dropdown("registrar_tablas", "si", ["si", "no"])
dbutils.widgets.dropdown("exportar_muestra", "si", ["si", "no"])
dbutils.widgets.text("metropolis_muestra", "Los Cabos,Guanajuato,Tecomán")

## 2. Configurar el data lake

In [ ]:
import os
import sys
from pathlib import Path

CATALOGO = "workspace"
ESQUEMA = "mccain_papa"
VOLUMEN = "datalake"
LAGO = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

os.environ["DATOS_MASIVOS_DATA"] = LAGO
os.environ["DATOS_MASIVOS_REPORTS"] = f"{LAGO}/reports"


def encontrar_raiz():
    """Buscamos hacia arriba la carpeta del repo (la que contiene src/)."""
    for candidata in [Path.cwd(), *Path.cwd().parents]:
        if (candidata / "src" / "processing" / "build_dataset.py").exists():
            return candidata
    raise RuntimeError("Abre este notebook desde el Git folder del repositorio.")


RAIZ = encontrar_raiz()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

from src.utils import rutas
from src.utils.config import cargar_config

CONFIG = cargar_config()
print("Data lake:", rutas.DATA)

## 3. Una llamada ejecuta el flujo completo

`leer_configuracion` → `ejecutar_pipeline`. Cada etapa imprime una línea JSON con su estado,
duración y error; el resumen queda en `reports/tables/registro_pipeline.json`. Si una etapa falla,
las siguientes no corren y la celda termina en error (así un Job aparece como fallido).

In [ ]:
from src import pipeline

argumentos = ["--fuente", "indice"]
if dbutils.widgets.get("descargar") == "no":
    argumentos.append("--sin-descarga")

config = pipeline.leer_configuracion(argumentos)
resultado = pipeline.ejecutar_pipeline(config)
print("Ejecución completada:", resultado["run_id"])

## 4. ¿El modelo sirve? Validación temporal

Entrenamos con 2015→2019 y 2019→2022 y probamos con 2022→2026, sin ver el futuro. Lo comparamos
con dos referencias: **persistencia** (cada municipio repite su crecimiento anterior) y
**promedio** (todos crecen igual). MAE en puntos porcentuales de crecimiento anual por municipio;
Spearman compara el orden de las metrópolis (1 = mismo orden).

In [ ]:
import pandas as pd

metricas = pd.read_csv(rutas.TABLAS / "modelo_metricas.csv")
display(metricas)

In [ ]:
from IPython.display import Image

Image(filename=str(rutas.FIGURAS / "importancia_variables.png"))

## 5. Heatmap interactivo

Haz clic en una metrópoli (punto o fila) para ver sus números e insights. El mismo archivo queda en `reports/figures/mapa_potencial.html`.

In [ ]:
displayHTML((rutas.FIGURAS / "mapa_potencial.html").read_text(encoding="utf-8"))

## 6. Top 15 por potencial incremental

In [ ]:
import json

potencial = pd.read_csv(rutas.TABLAS / "potencial_metropolis.csv")
mercado = json.loads((rutas.TABLAS / "mercado_parametros.json").read_text())
print(f"Mercado {mercado['anio_importaciones']}: {mercado['kg_por_restaurante']:,.0f} kg por restaurante, "
      f"US${mercado['usd_kg']:.2f}/kg")
columnas = ["ranking_potencial", "metropoli", "potencial_incremental_usd", "potencial_proyectado_usd",
            "crecimiento_esperado", "crecimiento_anual", "restaurantes_hoy",
            "restaurantes_proyectados", "ranking", "indice"]
display(potencial[columnas].head(15))

## 7. Tablas Gold en el catálogo

In [ ]:
TABLAS_CATALOGO = {
    "potencial_metropolis": "Potencial de mercado por metropoli: crecimiento esperado (XGBoost) y US$",
    "panel_municipios": "Restaurantes, cadenas y hoteles por municipio metropolitano y version del DENUE",
    "denue_rechazos": "Establecimientos del DENUE rechazados, con motivos_rechazo",
    "indice_inversion": "Indice de inversion por zona metropolitana (0-100) y sus componentes",
}

if dbutils.widgets.get("registrar_tablas") == "si":
    for tabla, descripcion in TABLAS_CATALOGO.items():
        ruta = rutas.PROCESSED / tabla
        if not ruta.exists():
            print(f"(sin {tabla})")
            continue
        nombre = f"{CATALOGO}.{ESQUEMA}.{tabla}"
        (spark.read.parquet(str(ruta)).write.mode("overwrite")
         .option("overwriteSchema", "true").saveAsTable(nombre))
        spark.sql(f"COMMENT ON TABLE {nombre} IS '{descripcion}'")
        print(f"{nombre:50s} {spark.table(nombre).count():>10,} filas")

In [ ]:
display(spark.sql(f"""
    SELECT ranking_potencial, metropoli,
           ROUND(potencial_incremental_usd / 1e6, 2) AS incremental_musd,
           ROUND(crecimiento_esperado, 1)            AS crec_esperado_pct,
           ROUND(crecimiento_anual, 1)               AS crec_2019_2026_pct,
           ranking                                   AS ranking_indice
    FROM {CATALOGO}.{ESQUEMA}.potencial_metropolis
    ORDER BY ranking_potencial
    LIMIT 10
"""))

## 8. Paquete para el repositorio

Juntamos en un solo zip lo que va al repositorio: las tablas de resultados, el heatmap, las
gráficas y (si `exportar_muestra = si`) una **muestra real** chica del DENUE y Comtrade para que
el pipeline corra sin internet. Descárgalo desde **Catalog → workspace → mccain_papa → datalake →
reports → entrega_final.zip → Download**.

In [ ]:
import shutil
import zipfile

from src.ingestion.muestra_indice import exportar_muestra_indice

ENTREGA = Path(LAGO) / "reports" / "entrega"
shutil.rmtree(ENTREGA, ignore_errors=True)
(ENTREGA / "tables").mkdir(parents=True)
(ENTREGA / "figures").mkdir(parents=True)
for nombre in ["potencial_metropolis.csv", "modelo_metricas.csv", "mercado_parametros.json",
               "indice_inversion.csv", "validacion.csv", "validacion.md", "registro_pipeline.json"]:
    if (rutas.TABLAS / nombre).exists():
        shutil.copy(rutas.TABLAS / nombre, ENTREGA / "tables" / nombre)
for ruta in rutas.FIGURAS.glob("*"):
    shutil.copy(ruta, ENTREGA / "figures" / ruta.name)

if dbutils.widgets.get("exportar_muestra") == "si":
    nombres = [n.strip() for n in dbutils.widgets.get("metropolis_muestra").split(",") if n.strip()]
    municipios = spark.read.parquet(str(rutas.PROCESSED / "metropolis_municipios"))
    print(exportar_muestra_indice(spark, nombres, rutas.INTERIM / "denue", municipios,
                                  rutas.RAW / "comtrade", ENTREGA / "sample_indice"))

zip_final = Path(LAGO) / "reports" / "entrega_final.zip"
with zipfile.ZipFile(zip_final, "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in sorted(ENTREGA.rglob("*")):
        if ruta.is_file():
            archivo.write(ruta, ruta.relative_to(ENTREGA))
print(f"{zip_final}  ({zip_final.stat().st_size / 1024**2:.1f} MB)")

## 9. Programarlo como Job (clase de automatización)

1. **Jobs & Pipelines → Create → Job**. Tarea tipo **Notebook**, *Path* = este notebook, cómputo
   **Serverless**.
2. Parámetros con los mismos nombres que los widgets: `descargar = si`, `registrar_tablas = si`,
   `exportar_muestra = no`.
3. **Run now** y revisar en la salida las líneas JSON de cada etapa y `RESUMEN_PIPELINE`.
4. Para programarlo: **Schedule**, por ejemplo el día 1 de cada mes a las 09:00
   *America/Mexico_City*, porque INEGI publica el DENUE por versiones y Comtrade mes por mes.
   Dejarlo en pausa al terminar la práctica.

Repetir la corrida no duplica nada: cada etapa escribe en modo `overwrite` y la descarga usa la
caché, así que el mismo insumo produce el mismo resultado (idempotencia).

## 10. Conclusión

1. **El índice dice dónde está la oportunidad hoy; el modelo dice dónde va a crecer.** La tabla
   `potencial_metropolis` junta los dos para cada metrópoli.
2. **El modelo se evalúa contra referencias simples** en un periodo que no vio. Si no les gana,
   lo reportamos así y usamos la proyección solo como escenario.
3. **El potencial en US$ es tamaño de mercado**, construido con datos abiertos; para hablar de
   rentabilidad faltan precios de venta y costos internos de McCain.